# 📏 Notebook 05 — Production Baseline Benchmarking

**Project:** Demand Forecasting — Corporación Favorita Grocery Sales  
**Author:** Senior ML Engineer  
**Purpose:** Establish baseline benchmark scores before advanced model training (Notebook 06)  
**Environment:** Google Colab / Local (VS Code, Jupyter)  

---

## 🎯 Objective

This notebook is **NOT** the final model. It serves three purposes:

1. **Validate** that the Feature Store from Notebook 04 produces learnable signals.
2. **Establish** baseline benchmark scores that every advanced model must beat.
3. **Build** a professional evaluation report with metrics, charts, and exported artifacts.

> **Notebook 06** will contain: LightGBM, CatBoost, XGBoost, Hyperparameter Tuning.

## ⚡ Memory & Execution Strategy

| Design Decision | Implementation |
|----------------|----------------|
| **Chronological Subsetting** | `df.tail(10_000_000)` — recent 10M rows only |
| **Zero Random Sampling** | Strict chronological order preserved everywhere |
| **Zero Dtype Recasting** | Feature Store already optimized (`float32`, `uint8`, `category`) |
| **Sequential Training** | Train → Evaluate → Save → Delete → GC after every model |
| **Error Resilience** | Every model wrapped in `try/except` — one failure never crashes the notebook |
| **RAM Monitoring** | Memory usage reported after every major operation |

## 🏗️ Pipeline Architecture

```
feature_store.parquet (Notebook 04 Output — 86.9M rows, ~5.8 GB on disk)
        ↓
Chronological Tail Subsetting → Recent 10,000,000 Rows (~85% RAM Reduction)
        ↓
Automatic Feature Detection (numeric, categorical, boolean, leakage columns)
        ↓
Strict Out-of-Time Train/Val Split (Val: 2017-08-01 → end)
        ↓
┌──────────────────────────────────────────────────────────────────────────┐
│ SEQUENTIAL BASELINE ENGINE (Train → Eval → Save → Delete → GC)         │
│                                                                          │
│   1. Mean Baseline         ──► Evaluate ──► Log ──► GC                  │
│   2. Median Baseline       ──► Evaluate ──► Log ──► GC                  │
│   3. Seasonal Naive (t-7)  ──► Evaluate ──► Log ──► GC                  │
│   4. Linear Regression     ──► Evaluate ──► Save ──► GC                 │
│   5. Ridge Regression      ──► Evaluate ──► Save ──► GC                 │
│   6. Lasso Regression      ──► Evaluate ──► Save ──► GC                 │
│   7. ElasticNet            ──► Evaluate ──► Save ──► GC                 │
│   8. Random Forest         ──► Evaluate ──► Save ──► GC                 │
└──────────────────────────────────────────────────────────────────────────┘
        ↓
Export: baseline_results.csv, metrics.json, benchmark_summary.csv, feature_list.txt
        ↓
Diagnostic Visualization Suite (5 High-DPI Charts)
```

## 📊 Evaluation Metrics

| Metric | Formula | Purpose |
|--------|---------|--------|
| **RMSLE** | $\sqrt{\frac{1}{n}\sum(\log(1+\hat{y}) - \log(1+y))^2}$ | Primary competition metric |
| **RMSE** | $\sqrt{\frac{1}{n}\sum(\hat{y} - y)^2}$ | Scale-sensitive error |
| **MAE** | $\frac{1}{n}\sum|\hat{y} - y|$ | Interpretable absolute error |
| **MAPE** | $\frac{100}{n}\sum|\frac{\hat{y} - y}{y}|$ | Percentage error (excludes zeros) |
| **R²** | $1 - \frac{SS_{res}}{SS_{tot}}$ | Variance explained |

| Specification | Detail |
|---------------|--------|
| **Inputs** | `01_Dataset/features/feature_store.parquet` |
| **Outputs** | `baseline_results.csv`, `metrics.json`, `benchmark_summary.csv`, Models, Charts |
| **Previous Notebook** | `04_feature_engineering.ipynb` |
| **Next Notebook** | `06_model_training.ipynb` |

---


## 1️⃣ Environment Setup & Project Bootstrap

Detects execution environment (Google Colab vs Local) and configures project paths.


In [ ]:
# ============================================================
# 1. Environment Bootstrap (Colab + Local)
# ============================================================
import os, sys
from pathlib import Path

# 1. Google Colab: Mount Drive
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

# 2. Find project root (checks Drive paths + local paths)
POSSIBLE_ROOTS = [
    Path('/content/drive/MyDrive/Demand-Forecasting-System'),
    Path('/content/drive/MyDrive/NTI/Demand-Forecasting-System'),
    Path('/content/drive/MyDrive/Colab Notebooks/Demand-Forecasting-System'),
    Path.cwd(),
    Path.cwd().parent,
]

PROJECT_ROOT = None
for p in POSSIBLE_ROOTS:
    if p.exists() and (p / 'config.py').exists():
        PROJECT_ROOT = p.resolve()
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        '❌ Project root with config.py not found.\n'
        'Colab: Make sure the folder is in your Google Drive.\n'
        'Local: Run the notebook from inside the project directory.'
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

ENV = 'Google Colab' if 'google.colab' in sys.modules else 'Local'
print('=' * 60)
print(f'📁 Project Root : {PROJECT_ROOT}')
print(f'📂 Working Dir  : {os.getcwd()}')
print(f'🖥️  Runtime      : {ENV}')
print('✅ Bootstrap OK')
print('=' * 60)


## 2️⃣ Imports & Configuration

Imports all required libraries, configures Matplotlib aesthetics, and sets up output directories.


In [ ]:
# TODO: Implement your code here


## 3️⃣ Load Feature Store

### 🎯 Purpose
Load the Feature Store generated by Notebook 04 via `utils.load_feature_store()`.

### ⚡ Memory Strategy
- Direct Parquet load — **zero dtype recasting** (Feature Store is already optimized).
- Immediately take `tail(BASELINE_ROWS)` to reduce RAM by ~85%.
- Delete the full DataFrame reference immediately after subsetting.


In [ ]:
# TODO: Implement your code here


## 4️⃣ Automatic Feature Detection & Leakage Prevention

### 🎯 Purpose
Automatically classify every column by type (target, date, ID, categorical, boolean, numeric) and exclude leakage/non-feature columns.

### 🚫 Excluded Columns
| Column | Reason |
|--------|--------|
| `date` | Temporal identifier — not a predictive feature |
| `id` | Row identifier — no predictive value |
| `unit_sales` | Target variable — must be separated |
| String categoricals | Already frequency-encoded in Notebook 04 |


In [ ]:
# TODO: Implement your code here


## 5️⃣ Chronological Out-of-Time Train / Validation Split

### 🎯 Purpose
Split the dataset using a **strict temporal cutoff** to prevent data leakage.

### ⚠️ Why Random Split is Forbidden
In time-series forecasting, random splits leak future information into training data, producing
over-optimistic scores that **fail catastrophically in production**.

| Set | Condition | Purpose |
|-----|-----------|--------|
| **Training** | $t < \text{2017-08-01}$ | Learn historical patterns |
| **Validation** | $t \geq \text{2017-08-01}$ | Simulate future prediction (matches Kaggle test horizon) |


In [ ]:
# TODO: Implement your code here


### 📊 5.1 Visualization: Temporal Train / Validation Split


In [ ]:
# ── 5.1 Temporal Split Visualization ──
# TODO: Implement your code here


## 6️⃣ Evaluation Framework & Metric Logger

### 🎯 Purpose
Central evaluation function that computes all metrics, saves models, logs results, and manages memory.

### 📊 Metrics Computed
| Metric | Description |
|--------|------------|
| RMSLE | Root Mean Squared Log Error (primary competition metric) |
| RMSE | Root Mean Squared Error |
| MAE | Mean Absolute Error |
| MAPE | Mean Absolute Percentage Error |
| R² | Coefficient of Determination |
| Train Time | Seconds to fit the model |
| Pred Time | Seconds to generate predictions |
| Model Size | Saved model file size in MB |


In [ ]:
# TODO: Implement your code here


## 7️⃣ Baseline Model Training & Evaluation

### 🏗️ Execution Protocol
Each model follows the same strict sequence:

1. **Train** the model (timed)
2. **Predict** on validation set (timed)
3. **Evaluate** all metrics
4. **Save** model to disk (if applicable)
5. **Delete** model object and predictions from RAM
6. **Garbage Collect** to reclaim memory

> ⚠️ Every model is wrapped in `try/except`. If one model fails, the notebook continues execution.


### 7️⃣.1 Baseline 1 — Global Mean

Predicts the global average of training sales: $\hat{y} = \bar{y}_{train}$  
Establishes the **absolute simplest lower bound**.


In [ ]:
# ── Baseline 1: Global Mean ──
# TODO: Implement your code here


### 7️⃣.2 Baseline 2 — Global Median

Predicts the global median: $\hat{y} = \text{Median}(y_{train})$  
More robust to extreme promotional sales spikes than the mean.


In [ ]:
# ── Baseline 2: Global Median ──
# TODO: Implement your code here


### 7️⃣.3 Baseline 3 — Seasonal Naive ($t-7$)

Predicts sales equal to the same day last week: $\hat{y}_t = y_{t-7}$  
Captures weekly seasonality patterns (e.g., weekday vs weekend demand cycles).


In [ ]:
# ── Baseline 3: Seasonal Naive (t-7) ──
# TODO: Implement your code here


### 7️⃣.4 Baseline 4 — Linear Regression (OLS)

Unregularized ordinary least squares trained on $\log(1+y)$ transformed target.  
Establishes the parametric linear baseline.


In [ ]:
# ── Baseline 4: Linear Regression ──
# TODO: Implement your code here


### 7️⃣.5 Baseline 5 — Ridge Regression ($L_2$)

Adds $L_2$ penalty ($\alpha = 1.0$) to stabilize coefficients against multicollinearity
in highly correlated lag and rolling window features.


In [ ]:
# ── Baseline 5: Ridge Regression ──
# TODO: Implement your code here


### 7️⃣.6 Baseline 6 — Lasso Regression ($L_1$)

Adds $L_1$ penalty ($\alpha = 0.01$) to enforce coefficient sparsity,
automatically zeroing out weak or redundant features.


In [ ]:
# ── Baseline 6: Lasso Regression ──
# TODO: Implement your code here


### 7️⃣.7 Baseline 7 — ElasticNet ($L_1 + L_2$)

Combines $L_1$ sparsity and $L_2$ grouping penalties ($\alpha = 0.01, l_1\_ratio = 0.5$).  
Effective when features are correlated in groups (e.g., multiple rolling window features).


In [ ]:
# ── Baseline 7: ElasticNet ──
# TODO: Implement your code here


### 7️⃣.8 Baseline 8 — Random Forest (Small)

Non-linear tree ensemble baseline lower bound.  
Deliberately kept small ($n\_estimators = 50$, $max\_depth = 10$) for RAM safety and speed.


In [ ]:
# ── Baseline 8: Random Forest (Lightweight) ──
# TODO: Implement your code here


## 8️⃣ Master Benchmark Results & Export

### 🎯 Purpose
Compile all baseline metrics into a master results table sorted by **RMSLE** (primary competition metric).

### 📦 Exported Artifacts
| File | Description |
|------|------------|
| `baseline_results.csv` | Full results table with all metrics |
| `metrics.json` | Machine-readable metrics dictionary |
| `benchmark_summary.csv` | Condensed summary for quick comparison |
| `feature_list.txt` | List of features used for training |


In [ ]:
# TODO: Implement your code here


## 9️⃣ Diagnostic Visualization Suite

### 🎯 Purpose
Generate presentation-quality diagnostic charts for the graduation report.

All figures are saved at **200 DPI** to `output/05_baseline_models/plots/` and closed immediately to free memory.


### 📊 9.1 Model Error Metrics Comparison


In [ ]:
# ── 9.1 Model Error Metrics Comparison (RMSLE, RMSE, MAE, R²) ──
# TODO: Implement your code here


### 📊 9.2 Operational Metrics Comparison


In [ ]:
# ── 9.2 Operational Metrics (Training Time, Prediction Time, Model Size) ──
# TODO: Implement your code here


### 📊 9.3 MAPE Comparison


In [ ]:
# ── 9.3 MAPE Comparison ──
# TODO: Implement your code here


### 📊 9.4 Actual vs Predicted & Residual Diagnostics (Best Model)

Loads the best-performing baseline model and generates:
1. **Actual vs Predicted Scatter Plot** — ideal predictions fall on $y = x$ line.
2. **Residual Error Distribution** — centered at zero indicates unbiased predictions.


In [ ]:
# ── 9.4 Actual vs Predicted & Residual Diagnostics ──
# TODO: Implement your code here


### 📊 9.5 Feature Correlation Heatmap

Visualizes Pearson correlation between the target and key features.


In [ ]:
# ── 9.5 Feature Correlation Heatmap ──
# TODO: Implement your code here


## 🔟 Final Cleanup & Execution Summary


In [ ]:
# TODO: Implement your code here


---

## ✅ Completion Checklist

| # | Requirement | Status |
|---|-------------|--------|
| 1 | Direct Parquet Load via `utils.load_feature_store()` | ✅ |
| 2 | Chronological Subset Selection (Last 10M rows, NO random sampling) | ✅ |
| 3 | Automatic Feature Detection & Leakage Prevention | ✅ |
| 4 | Strict Out-of-Time Chronological Train/Val Split | ✅ |
| 5 | 8 Baseline Models Trained & Evaluated | ✅ |
| 6 | RMSLE, RMSE, MAE, MAPE, R² Metrics Computed | ✅ |
| 7 | Training Time, Prediction Time, Model Size Measured | ✅ |
| 8 | `try/except` Error Handling Around Every Model | ✅ |
| 9 | RAM Monitoring After Every Major Operation | ✅ |
| 10 | `del` + `gc.collect()` Memory Cleanup After Every Model | ✅ |
| 11 | `plt.close(fig)` After Every Figure Save | ✅ |
| 12 | `baseline_results.csv` Exported | ✅ |
| 13 | `metrics.json` Exported | ✅ |
| 14 | `benchmark_summary.csv` Exported | ✅ |
| 15 | `feature_list.txt` Exported | ✅ |
| 16 | 5 Presentation-Grade Diagnostic Charts Saved | ✅ |
| 17 | Saved Models in `output/05_baseline_models/baseline_models/` | ✅ |

**➡️ Next:** Open `06_model_training.ipynb` for LightGBM, CatBoost, XGBoost training.
